In [ ]:
"""
This code will be my attempt in fully building and implementing a PINN in order to model non-linear partial differential equations such as the advection-diffusion
equation and Burger's equation.
"""

In [21]:
# Need to overload some operators in order to do algebra!

class Vector:
    def __init__(self, *components):
        """
        Create a vector of arbitrary length.
        Example:
            Vector(1, 2)
            Vector(1, 2, 3, 4)
        """
        if len(components) == 0:
            self.components = (0,)
        else:
            if not all(isinstance(x, (int, float)) for x in components):
                raise TypeError("All components must be numbers")
            self.components = tuple(components)

    def __repr__(self):
        return f"Vector{self.components}"

    def __len__(self):
        return len(self.components)

    def __getitem__(self, index):
        return self.components[index]

    # Vector addition
    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        
        if len(self) != len(other):
            raise ValueError("Vectors must have same dimension")

        return Vector(*(a + b for a, b in zip(self.components, other.components)))
    
    # Scalar multiplication
    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        
        return Vector(*(scalar * x for x in self.components))

    # Allow scalar * vector
    def __rmul__(self, scalar):
        return self.__mul__(scalar)

    # Dot product using @
    def __matmul__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        
        if len(self) != len(other):
            raise ValueError("Vectors must have same dimension")

        return sum(a * b for a, b in zip(self.components, other.components))

    # Magnitude
    def norm(self):
        return sum(x**2 for x in self.components) ** 0.5
    
    # Also need to add hyper-dual operations as well for numerical differentiation we will use for backpropagation
    # Addition is the same

    def hyp_mult(self, other):

        if not isinstance(other, Vector):
            return NotImplemented

        if len(self) != 4 or len(other) != 4:
            raise ValueError("Hyper-dual vectors must have dimension 4")

        a, b, c, d = self.components
        e, f, g, h = other.components

        return Vector(
            a * e,
            a * f + b * e,
            a * g + c * e,
            a * h + b * g + c * f + d * e
        )
    
    def hyp_div(self):

        if len(self) != 4:
            raise ValueError("Hyper-dual vectors must have dimension 4")
        
        a, b, c, d = self.components

        if a == 0:
            raise ZeroDivisionError("Real component cannot be zero")

        return Vector(1/a, -b/a**2, -c/a**2, (2*b*c / a**3) - (d / a**2))
    
    def hyp_norm(self):

        if len(self) != 4:
            raise ValueError("Hyper-dual vectors must have dimension 4")
        
        a, b, c, d = self.components

        return ((a)**(2))**(0.5)



 
# Tests
v1 = Vector(1, 2)
v2 = Vector(3, 4)

v3 = Vector(1, 2, 3)
v4 = Vector(4, 5, 6)

v5 = Vector(1, 2, 3, 4)
v6 = Vector(5, 6, 7, 8)

print(v1)
print(v2)

print(v3)
print(v4)

print(v1 + v2)
print(v1 + -1*v2)
print(v3 @ v4)
print(v3.norm())
print(v5.hyp_mult(v6))
print(v5.hyp_div())
print(v5.hyp_norm())




Vector(1, 2)
Vector(3, 4)
Vector(1, 2, 3)
Vector(4, 5, 6)
Vector(4, 6)
Vector(-2, -2)
32
3.7416573867739413
Vector(5, 16, 22, 60)
Vector(1.0, -2.0, -3.0, 8.0)
1.0
